# Gradient Boosting | Banking loan default

Gradient Boosting adds shallow trees that reduce the chosen loss. It builds an additive model step by step.

**Goal:** Predict `loan_default` (1 = default, 0 = no default). All three notebooks use the same CSV, 80/20 stratified split, random seed, preprocessing and metrics. Results depend on the supplied data; no accuracy is promised. The example data is for teaching, not real lending decisions.

## 1. Setup

Python 3.11 is suitable. Install these packages once in your notebook environment. The first command is intentionally commented so it does not reinstall packages each run.

In [ ]:
# %pip install pandas numpy matplotlib seaborn scikit-learn xgboost
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix, RocCurveDisplay
from sklearn.ensemble import GradientBoostingClassifier

## 2. Read and inspect

Place `Banking_Loan_Default_Classification.csv` beside this notebook. Each row represents one loan applicant; `loan_default` is the answer the model learns to predict.


In [ ]:
df = pd.read_csv("Banking_Loan_Default_Classification.csv")
print("Shape:", df.shape)
display(df.head())
print("Missing values:")
display(df.isna().sum().to_frame("missing"))
print("Target distribution:")
display(df["loan_default"].value_counts().sort_index().to_frame("count"))

## 3. Separate inputs and target; split before fitting preprocessing

`stratify=y` keeps the default proportion similar in training and test data. Fit imputers and encoding **only on training rows** through the pipeline, so test information cannot leak into training.

In [ ]:
X = df.drop(columns="loan_default")
y = df["loan_default"]
assert set(y.unique()) == {0, 1}, "Expected 0/1 default labels"
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Default rate, train:", round(y_train.mean(), 3))
print("Default rate, test:", round(y_test.mean(), 3))

## 4. Prepare numerical and categorical columns

Numbers use median imputation. Text categories use most frequent imputation followed by one hot encoding. Unknown categories at prediction time are ignored safely.

In [ ]:
numeric_columns = X.select_dtypes(include="number").columns.tolist()
categorical_columns = X.select_dtypes(exclude="number").columns.tolist()
print("Numeric:", numeric_columns)
print("Categorical:", categorical_columns)
numeric_pipe = Pipeline([("imputer", SimpleImputer(strategy="median"))])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])
preprocess = ColumnTransformer([
    ("numeric", numeric_pipe, numeric_columns),
    ("categorical", categorical_pipe, categorical_columns)
])

## 5. Train the model and display loss

`n_estimators=150` trains 150 boosting stages. `learning_rate=0.05` controls how much each new tree contributes; `max_depth=3` limits each tree’s depth. The model finishes fitting before the next cell prints and plots training loss after each stage. A falling training loss alone does not prove that the model performs better on new applicants.


In [ ]:
model = GradientBoostingClassifier(n_estimators=150, learning_rate=0.05, max_depth=3, random_state=42)
clf = Pipeline([("preprocess", preprocess), ("model", model)])
clf.fit(X_train, y_train)
print("Training complete")

In [ ]:
# Training loss after each boosting stage
loss_history = pd.DataFrame({
    "stage": np.arange(1, model.n_estimators_ + 1),
    "training_loss": clf.named_steps["model"].train_score_
})

#print(loss_history.to_string(index=False))

from IPython.display import clear_output, display
import time
#print(loss_history.to_string(index=False))
for stage in range(1, len(loss_history) + 1):
    clear_output(wait=True)
    display(loss_history.iloc[:stage].round(6))
    time.sleep(0.2)


plt.figure(figsize=(9, 5))
plt.plot(loss_history["stage"], loss_history["training_loss"], linewidth=2)
plt.xlabel("Boosting stage")
plt.ylabel("Training loss (deviance)")
plt.title("Gradient Boosting training loss after each stage")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 6. Predict and evaluate

For the positive class **default (1)**: precision = TP/(TP+FP), the fraction of predicted defaults that truly default; recall = TP/(TP+FN), the fraction of actual defaults found; F1 balances precision and recall. Accuracy measures all correct predictions / all cases. ROC AUC measures ranking across thresholds, not accuracy at one threshold.

In [ ]:
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]
scores = pd.DataFrame({
    "metric": ["Accuracy", "Precision (default=1)", "Recall (default=1)", "F1 (default=1)", "ROC AUC"],
    "value": [accuracy_score(y_test,y_pred), precision_score(y_test,y_pred,zero_division=0), recall_score(y_test,y_pred,zero_division=0), f1_score(y_test,y_pred,zero_division=0), roc_auc_score(y_test,y_prob)]
})
display(scores.round(3))
print(classification_report(y_test, y_pred, target_names=["No default (0)","Default (1)"], zero_division=0))

## 7. Confusion matrix

Rows are **actual** and columns are **predicted**. Top left = TN, top right = FP, bottom left = FN, bottom right = TP. In lending, a false negative means an actual default was predicted as no default.

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
cm_table = pd.DataFrame(cm, index=["Actual no default (0)","Actual default (1)"], columns=["Predicted no default (0)","Predicted default (1)"])
display(cm_table)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm_table, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion matrix")
plt.tight_layout()
plt.show()
print(f"TN={cm[0,0]}, FP={cm[0,1]}, FN={cm[1,0]}, TP={cm[1,1]}")

## 8. ROC curve

A curve closer to the upper-left corner indicates stronger separation. The diagonal represents random ranking. This does not choose a business threshold for you.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
RocCurveDisplay.from_predictions(y_test, y_prob, ax=ax, name="Gradient Boosting")
ax.plot([0,1], [0,1], "--", color="gray", label="Random ranking")
ax.legend()
plt.tight_layout()
plt.show()

## 9. Illustrative feature importance

These values describe what the fitted trees used, not causal effects. One hot columns are shown separately; importance definitions differ across boosting libraries, so do not compare raw values across notebooks.

In [ ]:
feature_names = clf.named_steps["preprocess"].get_feature_names_out()
importance = clf.named_steps["model"].feature_importances_
top_features = pd.Series(importance, index=feature_names).sort_values(ascending=False).head(12)
display(top_features.to_frame("importance"))
fig, ax = plt.subplots(figsize=(8, 5))
top_features.sort_values().plot.barh(ax=ax)
ax.set_title("Top 12 model features")
ax.set_xlabel("Model importance")
plt.tight_layout()
plt.show()

## 10. Predict one example and discuss

This example is drawn from the held-out test set. A probability is a model estimate, not a guarantee. To compare these three algorithms, run each notebook against the same CSV and compare the five metrics above; decide whether missing defaults (FN) or false alarms (FP) matter more.

In [ ]:
one_applicant = X_test.iloc[[0]]
print("Actual label:", y_test.iloc[0])
print("Predicted label:", clf.predict(one_applicant)[0])
print("Estimated default probability:", round(clf.predict_proba(one_applicant)[0,1], 3))
display(one_applicant)